# Lecture 02 · Tracking, callbacks and modifying PPO
**RLII_26 · Advanced Reinforcement Learning**


In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import pandas as pd

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "rl_zoo3" / "exp_manager.py").is_file())
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / "logs" / "lecture_02" / ".cache" / "matplotlib"))
print("Python:", sys.executable)


## 1 · W&B: account, team and project

- Register at [wandb.ai](https://wandb.ai) and join the course team with write access.
- The team identifier is `RL2_2026`; the demo project is `Lecture_02_demo`.
- Each training process creates a **run** containing its configuration and metrics.
- A **group** collects related runs, such as the seeds of one experiment.

[Shared course project](https://wandb.ai/RL2_2026/Lecture_02_demo)


In [ ]:
import wandb

wandb.login()


## 2 · From a Zoo command to tracked results

The repository's `train.py` calls `rl_zoo3.train.train()`. Zoo prepares the
experiment; SB3 owns the learning loop.

~~~text
train.py → rl_zoo3.train.train()
  ├── parse flags, choose the seed, initialize W&B with --track
  ├── ExperimentManager.setup_experiment()
  │     ├── read_hyperparameters(): YAML defaults + CLI overrides
  │     ├── create_callbacks(): evaluation, checkpoints, optional progress bar
  │     ├── create_envs(): training environments wrapped with Monitor
  │     └── construct SB3 PPO
  ├── ExperimentManager.learn() → model.learn(callback=callbacks)
  │     └── repeat: collect_rollouts() → dump_logs() → PPO.train()
  ├── save_trained_model(): final model
  └── W&B run.finish(): finish synchronization
~~~

A **callback** is an object that SB3 calls at defined points in this loop.
It can inspect the model, evaluate it, save data, or request that training stop.
Zoo passes a list of callbacks to `learn()`; SB3 combines them in a `CallbackList`.

| Source | Locate |
|---|---|
| [rl_zoo3/train.py](../../rl_zoo3/train.py) | `train()`, `wandb.init(...)` |
| [exp_manager.py](../../rl_zoo3/exp_manager.py) | `setup_experiment()`, `create_callbacks()`, `learn()` |
| [SB3 callbacks.py](../../.venv/lib/python3.12/site-packages/stable_baselines3/common/callbacks.py) | `BaseCallback`, `CallbackList`, `EvalCallback` |


### What happens during collection?

~~~text
learn(): on_training_start()
  collect_rollouts():
    buffer.reset()
    on_rollout_start()
    repeat:
      policy → env.step()
      on_step() → CheckpointCallback / EvalCallback
      buffer.add(...)
    compute_returns_and_advantage()
    on_rollout_end()
  dump_logs()
  PPO.train()
  ... next rollout ...
on_training_end()
~~~

The public callback methods update `n_calls`, `num_timesteps`, and references
to the training state, then call hooks such as `_on_step()`.
Returning `False` from `_on_step()` stops training.

| Callback in Zoo | When it acts | What it does |
|---|---|---|
| `EvalCallback` | Every `--eval-freq` training transitions | Calls `evaluate_policy()` on separate environments; logs evaluation return/length, writes `evaluations.npz`, saves `best_model.zip` on improvement |
| `CheckpointCallback` | Every `--save-freq` training transitions | Saves a model snapshot; it does not calculate performance metrics |
| `SaveVecNormalizeCallback` (Zoo) | When `EvalCallback` finds a new best result | Saves normalization statistics if `VecNormalize` is present |
| `ProgressBarCallback` | Each step, when `--progress` is enabled | Updates the progress display |
| `CallbackList` | Each callback event | Forwards the event to the registered callbacks |

`evaluate_policy()` repeatedly calls `model.predict()` and `env.step()` without
updating parameters. For LunarLander, Zoo uses deterministic actions and averages
undiscounted episode returns over `--eval-episodes` episodes.

One vector step advances all four demo environments. Zoo therefore converts
`--eval-freq 20480` to `EvalCallback(eval_freq=5120)`.
[Callback documentation](https://stable-baselines3.readthedocs.io/en/v2.9.0/guide/callbacks.html)


### Which metrics are recorded, and when?

| Metric / data | Producer | When |
|---|---|---|
| Episode return and length | `Monitor` environment wrapper | At the end of each training episode; also writes `*.monitor.csv` |
| `rollout/ep_rew_mean`, `rollout/ep_len_mean` | `OnPolicyAlgorithm.dump_logs()` | At the configured rollout logging interval, averaging recent completed episodes |
| `train/policy_gradient_loss`, `train/value_loss`, `train/entropy_loss`, `train/approx_kl`, `train/clip_fraction` | `PPO.train()` | Records statistics after updating on a rollout |
| `eval/mean_reward`, `eval/mean_ep_length` | `EvalCallback._on_step()` | After each scheduled evaluation; immediately flushes the logger |

`Monitor` is a wrapper. PPO's update metrics are recorded by the algorithm
itself. `EvalCallback` supplies the separate evaluation metrics.

`logger.record()` stores values; `logger.dump()` writes them to the configured
outputs. In PPO's loop, regular dumping happens before `train()`, so update
statistics become visible at a later dump. An evaluation can also trigger a dump.

~~~text
Monitor / PPO.train() / EvalCallback
              ↓
      logger.record() → logger.dump()
              ↓
     TensorBoard event files
              ↓
     W&B sync_tensorboard=True
~~~

Zoo initializes W&B to synchronize those event files. It does not need a separate
W&B callback for this pipeline.

Sources: [on_policy_algorithm.py](../../.venv/lib/python3.12/site-packages/stable_baselines3/common/on_policy_algorithm.py),
[ppo.py](../../.venv/lib/python3.12/site-packages/stable_baselines3/ppo/ppo.py),
[logger.py](../../.venv/lib/python3.12/site-packages/stable_baselines3/common/logger.py).


### Configuration and tracking flags

The LunarLander entry in `hyperparams/ppo.yml` supplies the defaults.
The short demo overrides the rollout size, number of epochs, and training budget.

| Flag | Meaning |
|---|---|
| `--track` | Initialize W&B and synchronize TensorBoard metrics |
| `--wandb-project-name` | Project receiving the runs |
| `--wandb-entity` | Team owning that project |
| `--wandb-group` | Group these related training runs |
| `--tensorboard-log` | Local TensorBoard directory when running without `--track` |
| `-f` | Local model, evaluation and configuration output root |

With `--track`, Zoo sets an absolute TensorBoard path under `runs/<run-name>/`.
The demo runs from `logs/lecture_02/demo/`, so all its files stay under `logs/`.


### Two PPO runs

Each seed requests 1,000,000 transitions. Evaluation runs every 20,480 transitions,
and checkpoints are saved every 40,960 transitions.

Logging runs once per rollout (1,024 transitions). PPO finishes complete rollouts,
so each run ends at 1,000,448 transitions.


In [ ]:
subprocess.run(
    ["bash", str(ROOT / "course" / "lecture_02" / "scripts" / "train_ppo_example.sh")],
    cwd=ROOT,
    env={**os.environ, "PYTHON": sys.executable},
    check=True,
)


## 3 · TensorBoard and W&B in the browser

The next cell starts TensorBoard in the background and displays its browser link.
It selects an available port automatically. Open that link and the
[W&B project](https://wandb.ai/RL2_2026/Lecture_02_demo) in the browser.

| Metric | Interpretation |
|---|---|
| `rollout/ep_rew_mean` | Mean return over recent completed training episodes |
| `eval/mean_reward` | Mean return from the separate evaluation environment |
| `train/policy_gradient_loss` | PPO's policy objective, expressed as a loss |
| `train/value_loss` | Error between value predictions and return targets |
| `train/entropy_loss` | Negative policy entropy; more negative means more entropy |
| `train/approx_kl` | Estimated change from the rollout policy |
| `train/clip_fraction` | Fraction of probability ratios outside the clipping interval |

**TensorBoard:** select the two seed runs, compare scalar curves, and change the
smoothing slider. The horizontal axis counts training environment transitions.

**W&B:** filter by group `lecture02_demo`, inspect `seed` and
`saved_hyperparams`, and select the same metrics. Use `global_step` as the
horizontal axis. Watch new runs appear as other students execute the training cell.


In [ ]:
from tensorboard import program
from IPython.display import Markdown, display

tensorboard = program.TensorBoard()
tensorboard.configure(argv=[
    "tensorboard", "--logdir", str(ROOT / "logs" / "lecture_02" / "demo" / "runs"),
    "--port", "0",
])
display(Markdown(f"[Open TensorBoard in your browser]({tensorboard.launch()})"))


## 4 · From last week's PPO objective to discounting transitions

Last week, we wrote the policy objective as

$$
J(\theta)=\mathbb E\left[\sum_{t=0}^{T-1}\gamma^t R_{t+1}\right].
$$

PPO alternates between collecting a rollout and updating its policy using that
rollout. Each stored transition contains the observation, action, old action
log probability, value estimate, advantage, and return target.

The advantage $\widehat A_t$ estimates how good the chosen action was relative
to the value of its state. Positive advantages encourage the action; negative
advantages discourage it. The probability ratio compares the current policy
with the policy that collected the data:

$$
r_t(\theta)=\frac{\pi_\theta(A_t\mid S_t)}
{\pi_{\theta_{\mathrm{old}}}(A_t\mid S_t)}.
$$

PPO's clipped policy loss for one transition is

$$
\ell_t^\pi(\theta)=-
\min\left(r_t(\theta)\widehat A_t,\,
\operatorname{clip}(r_t(\theta),1-\epsilon,1+\epsilon)\widehat A_t\right).
$$

SB3 normally averages these losses over uniformly selected rollout transitions.
It adds a value-prediction loss and a negative-entropy term, then calls
`loss.backward()` and `optimizer.step()`.


### Two places where discounting enters

SB3 already uses `gamma` when computing advantages and return targets.
There, future rewards are discounted relative to the **current transition**:

$$
G_t=R_{t+1}+\gamma R_{t+2}+\gamma^2 R_{t+3}+\cdots.
$$

The homework adds the factor $\gamma^t$ associated with the transition's position
relative to the **start of its episode**. It changes how strongly that transition
contributes to the update. It does not discount the rewards or advantages a second time.

Use $i$ to index a stored transition and $t_i$ for its episode time.
The first action of an episode has $t_i=0$, so its weight is one:

$$
w_i=\gamma^{t_i}.
$$

A rollout boundary is just a storage boundary. An episode can continue into the
next rollout, so its counter must continue too. Each vectorized environment
needs its own counter. `episode_start` marks an actual reset, including a
time-limit reset.

**Where this belongs:** `DiscountedRolloutBuffer.add()` stores the weights.
`reset()` clears rollout storage while keeping the episode counters.
The inherited `compute_returns_and_advantage()` continues to calculate the original targets.


### Apply the weights in the loss or in sampling

For $N$ transitions in a rollout, normalize the raw weights once:

$$
p_i=\frac{w_i}{\sum_{j=1}^{N}w_j},\qquad a_i=Np_i.
$$

The probabilities $p_i$ sum to one; the loss weights $a_i$ have mean one.
Let $\ell_i$ denote a transition's combined policy, value, and entropy loss.
The target weighted average is

$$
L=\sum_{i=1}^{N}p_i\ell_i.
$$

| Variant | How a minibatch is obtained | How its losses are averaged |
|---|---|---|
| Baseline | SB3's uniform permutation | Ordinary mean |
| Loss weighting | The same uniform permutation | Multiply each transition's loss by $a_i$, then take the mean |
| Weighted sampling | Draw $N$ indices per epoch with replacement using $p_i$ | Ordinary mean; sampling has already applied the weighting |

For loss weighting, a minibatch $B$ uses

$$
L_B=\frac{1}{|B|}\sum_{i\in B}a_i\ell_i.
$$

Use the full-rollout normalization even when processing a smaller minibatch.
Apply it to the policy, value, and entropy reductions. In weighted sampling,
a transition may appear several times or not at all in an epoch.

**Where this belongs:** `DiscountedRolloutBuffer.get()` selects indices and
prepares weights. `_get_samples()` keeps weights attached to their transitions.
`DiscountedPPO.train()` contains the three loss reductions.


### Local code for the assignment

Work in [scripts/discounted_ppo.py](scripts/discounted_ppo.py).
It provides local subclasses of SB3's `RolloutBuffer` and `PPO`.
The baseline mode works immediately; the two discounting modes contain TODOs.

| Method | Role in PPO | Your change |
|---|---|---|
| `OnPolicyAlgorithm.learn()` | Alternate collection and training | Inherited |
| `collect_rollouts()` | Step environments, handle timeouts, and call `buffer.add()` | Inherited |
| `DiscountedRolloutBuffer.reset()`, `add()` | Reset storage and store each transition | Track episode time and raw weights |
| `compute_returns_and_advantage()` | Compute GAE and return targets | Inherited |
| `swap_and_flatten()` | Turn step/environment axes into a transition axis | Use the same ordering for weights and observations |
| `DiscountedRolloutBuffer.get()`, `_get_samples()` | Select indices and construct minibatches | Sampling probabilities and minibatch weights |
| `DiscountedPPO.train()` | Compute losses and update parameters | Weight the three loss reductions |

The harness includes a local copy of SB3 2.9.0's `PPO.train()` so its losses can
be edited directly. Other learning behavior is inherited. Use the installed
[ppo.py](../../.venv/lib/python3.12/site-packages/stable_baselines3/ppo/ppo.py) and
[buffers.py](../../.venv/lib/python3.12/site-packages/stable_baselines3/common/buffers.py)
as references; edit the course file.

[course/lecture_02/scripts/train.py](scripts/train.py) registers `ppo_homework` and `ppo_silly`
in Zoo's `ALGOS` dictionary, then calls Zoo's existing `train()`.
`--conf-file hyperparams/ppo.yml` supplies the usual PPO settings.
Environment creation, callbacks, logging, and model saving remain the Zoo pipeline.


## 5 · Debug a deliberately silly modification

In [scripts/silly_ppo.py](scripts/silly_ppo.py), we reverse the advantage sign
before every PPO update:

~~~python
self.rollout_buffer.advantages *= -1
~~~

An action that had a positive advantage now gets a negative one.
This is deliberately a bad learning rule, but its intended behavior is easy to test.
The subclass changes this one operation and delegates the update to `super().train()`.


### Use breakpoints to check a prediction

Open [RLII_26.code-workspace](../../RLII_26.code-workspace) and select
**Lecture 02: Silly PPO (debug)**.

1. Set a breakpoint on `self.rollout_buffer.advantages *= -1`.
2. Press **F5**, then continue from the entry point.
3. Inspect `advantages_before` and `self.rollout_buffer.advantages`.
   Press **F10**: every sign should reverse.
4. Stop on `super().train()` and use **F11** to enter SB3's update.
   Inspect `rollout_data.advantages` at the policy-loss calculation.

| Workspace field | Meaning for this example |
|---|---|
| `program` | The course entry point that registers the local subclasses |
| `args` | Native Zoo flags; `--algo ppo_silly` selects this modification |
| `cwd`, `python` | Repository root and its Python environment |
| `justMyCode: false` | Step from the local subclass into SB3 |
| `stopOnEntry: true` | Pause before training starts |


## 6 · **TODO**: Assignment: discounted PPO

1. Complete the four TODO areas in `scripts/discounted_ppo.py`: episode weights,
   weighted sampling, full-rollout loss weights, and PPO's loss reductions.
   Keep `discounting="none"` as the ordinary PPO baseline.
2. Use **Lecture 02: Discounted PPO (debug)** to inspect your code. Start in
   baseline mode, then change its `discounting` argument to the variant you implemented.
3. Use one of the final cells to compare the three modes at
   $\gamma\in\{0.99,0.999\}$ with seeds **0, 1, 2**, one million transitions per run:
   **18 runs**. Keep other settings fixed, including `normalize_advantage=False`
   and `target_kl=None`.
4. Keep outputs under `logs/lecture_02/homework/` and track in
   `RL2_2026/RL2_2026`. Filter to your runs and compare
   `saved_hyperparams.discounting` and `saved_hyperparams.gamma` across seeds.
5. Submit your local harness, commands, W&B comparison, and a short interpretation.

The script uses `--algo ppo_homework` through the course entry point.
Choose a mode with the native flag `--hyperparams discounting:"'loss_weighted'"`.


## 7 · Test your implementation

After completing the TODOs, choose **one** of the following cells. Each trains
and evaluates all **18 homework configurations**: three modes, two discount
factors, and three seeds, with one million transitions per run. The runs execute
sequentially; these are the full homework experiments.

### Local runs

Models, evaluation results, and TensorBoard logs go to
`logs/lecture_02/homework/local/`. Compare `evaluations.npz` and the TensorBoard
curves across configurations. Training successfully does not by itself establish
that the weighting is correct; use the debugger to inspect the quantities described above.


In [ ]:
output = ROOT / "logs/lecture_02/homework/local"
output.mkdir(parents=True, exist_ok=True)

for discounting in ("none", "loss_weighted", "weighted_sampling"):
    for gamma in (0.99, 0.999):
        for seed in (0, 1, 2):
            subprocess.run([
                sys.executable, str(ROOT / "course/lecture_02/scripts/train.py"),
                "--algo", "ppo_homework",
                "--conf-file", str(ROOT / "hyperparams/ppo.yml"),
                "--env", "LunarLander-v3", "--n-timesteps", "1000000",
                "--seed", str(seed), "--log-interval", "1",
                "--eval-freq", "10000", "--eval-episodes", "5", "--n-eval-envs", "1",
                "--device", "cpu", "--num-threads", "1",
                "--hyperparams", "n_envs:16", "n_steps:1024", "batch_size:64",
                "n_epochs:2", f"gamma:{gamma}", "normalize_advantage:False",
                "target_kl:None", f"discounting:'{discounting}'",
                "--tensorboard-log", str(output / "runs"),
                "--uuid", "-f", ".",
            ], cwd=output, check=True)


### The same runs with W&B tracking

This cell uses the same configurations and additionally sends their metrics to
the [homework project](https://wandb.ai/RL2_2026/RL2_2026), in group
`lecture02_homework`. Log in using Section 1 first. Local files go to
`logs/lecture_02/homework/tracked/`.

Filter to your runs and compare `saved_hyperparams.discounting`,
`saved_hyperparams.gamma`, and `seed`, using `eval/mean_reward` against `global_step`.
This starts new runs; it does not upload runs from the previous cell.


In [ ]:
output = ROOT / "logs/lecture_02/homework/tracked"
output.mkdir(parents=True, exist_ok=True)

for discounting in ("none", "loss_weighted", "weighted_sampling"):
    for gamma in (0.99, 0.999):
        for seed in (0, 1, 2):
            subprocess.run([
                sys.executable, str(ROOT / "course/lecture_02/scripts/train.py"),
                "--algo", "ppo_homework",
                "--conf-file", str(ROOT / "hyperparams/ppo.yml"),
                "--env", "LunarLander-v3", "--n-timesteps", "1000000",
                "--seed", str(seed), "--log-interval", "1",
                "--eval-freq", "10000", "--eval-episodes", "5", "--n-eval-envs", "1",
                "--device", "cpu", "--num-threads", "1",
                "--hyperparams", "n_envs:16", "n_steps:1024", "batch_size:64",
                "n_epochs:2", f"gamma:{gamma}", "normalize_advantage:False",
                "target_kl:None", f"discounting:'{discounting}'",
                "--tensorboard-log", str(output / "runs"),
                "--track", "--wandb-project-name", "RL2_2026",
                "--wandb-entity", "RL2_2026", "--wandb-group", "lecture02_homework",
                "--uuid", "-f", ".",
            ], cwd=output, check=True)
